# DeepMind Weather Forecast JSON Generator

This Colab notebook generates `deepmind_forecast.json` for the global weather model comparison dashboard.

Default mode is a deterministic schema-valid forecast so the dashboard pipeline can run end to end immediately. Replace `run_deepmind_adapter()` with a real WeatherNext or GraphCast inference implementation when model weights and atmospheric input tensors are available.

## 1. Runtime Setup

Use a GPU runtime when running real DeepMind inference. The fallback generator runs on CPU.

In [ ]:
!python --version
!pip -q install numpy pandas xarray gcsfs zarr netCDF4

## 2. Configuration

Set the target coordinate and optional GitHub destination. If you want Colab to push the JSON back to GitHub, add a Colab secret named `GITHUB_TOKEN` with repo write access.

In [ ]:
LATITUDE = 25.03
LONGITUDE = 121.56
HOURS = 48
OUTPUT_PATH = "/content/deepmind_forecast.json"

GITHUB_OWNER = ""
GITHUB_REPO = ""
GITHUB_BRANCH = "main"
GITHUB_TARGET_PATH = "public/deepmind_forecast.json"

## 3. Optional Google Drive Mount

Mount Drive if you want to keep a copy under your Google account.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. DeepMind Adapter

WeatherNext open-source models and demo notebooks are published by Google, while WeatherNext 3 is described by Google as an operational service rather than an open-source model. GraphCast also has open-source code, but a real run requires model weights plus analysis or forecast initial-condition tensors.

Keep the output contract as a list of hourly timestamps and 2m temperature values in Celsius.

In [ ]:
from __future__ import annotations

import json
import math
from datetime import datetime, timedelta, timezone
from pathlib import Path


def run_deepmind_adapter(latitude: float, longitude: float, hours: int):
    """Return hourly 2m temperature predictions from WeatherNext or GraphCast.

    Replace this function with real model inference once these inputs are ready:
    - model package and checkpoint path
    - ERA5/GFS atmospheric initial-condition tensors
    - interpolation from model grid to the requested latitude/longitude

    Expected return shape:
    {
      "time": ["2026-09-09T00:00", ...],
      "temperature_2m": [26.5, ...]
    }
    """
    return None


def build_fallback_forecast(latitude: float, longitude: float, hours: int):
    generated_at = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
    local_hour_offset = round(longitude / 15)
    local_now = (generated_at + timedelta(hours=local_hour_offset)).replace(tzinfo=None)
    start = local_now.replace(hour=0)
    times = []
    temperatures = []

    for step in range(hours):
        timestamp = start + timedelta(hours=step)
        local_hour = timestamp.hour
        daytime_wave = math.sin(((local_hour - 7) / 24) * 2 * math.pi)
        synoptic_wave = math.sin((step / 18) * 2 * math.pi) * 0.4
        latitude_adjustment = max(-4.0, min(4.0, (25 - abs(latitude)) * 0.05))
        temperature = 29.4 + daytime_wave * 3.1 + synoptic_wave + latitude_adjustment

        times.append(timestamp.strftime("%Y-%m-%dT%H:00"))
        temperatures.append(round(temperature, 1))

    return {
        "time": times,
        "temperature_2m": temperatures,
    }


def generate_payload(latitude: float, longitude: float, hours: int):
    generated_at = datetime.now(timezone.utc).replace(microsecond=0)
    hourly = run_deepmind_adapter(latitude, longitude, hours)
    model_name = "Google DeepMind WeatherNext"

    if hourly is None:
        hourly = build_fallback_forecast(latitude, longitude, hours)
        model_name = "Google DeepMind WeatherNext placeholder"

    return {
        "model": model_name,
        "generated_at": generated_at.isoformat().replace("+00:00", "Z"),
        "latitude": latitude,
        "longitude": longitude,
        "hourly": hourly,
    }


payload = generate_payload(LATITUDE, LONGITUDE, HOURS)
Path(OUTPUT_PATH).write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"Wrote {OUTPUT_PATH}")
print(json.dumps(payload, ensure_ascii=False, indent=2)[:1200])

## 5. Save a Copy to Google Drive

In [ ]:
from pathlib import Path
import shutil

drive_output_dir = Path('/content/drive/MyDrive/weather-model-dashboard')
drive_output_dir.mkdir(parents=True, exist_ok=True)
drive_output_path = drive_output_dir / 'deepmind_forecast.json'
shutil.copyfile(OUTPUT_PATH, drive_output_path)
print(f"Saved Drive copy: {drive_output_path}")

## 6. Optional GitHub Push

This updates `public/deepmind_forecast.json` in your dashboard repository. Add `GITHUB_TOKEN` in Colab Secrets before running this cell.

In [ ]:
import base64
import requests

try:
    from google.colab import userdata
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    github_token = None

if not (github_token and GITHUB_OWNER and GITHUB_REPO):
    print('Skipping GitHub push. Set GITHUB_TOKEN, GITHUB_OWNER, and GITHUB_REPO to enable it.')
else:
    api_url = f'https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/contents/{GITHUB_TARGET_PATH}'
    headers = {
        'Authorization': f'Bearer {github_token}',
        'Accept': 'application/vnd.github+json',
        'X-GitHub-Api-Version': '2022-11-28',
    }
    existing = requests.get(api_url, headers=headers, params={'ref': GITHUB_BRANCH})
    sha = existing.json().get('sha') if existing.status_code == 200 else None
    content = Path(OUTPUT_PATH).read_bytes()
    body = {
        'message': 'Update DeepMind forecast JSON',
        'branch': GITHUB_BRANCH,
        'content': base64.b64encode(content).decode('ascii'),
    }
    if sha:
        body['sha'] = sha
    response = requests.put(api_url, headers=headers, json=body)
    response.raise_for_status()
    print('Updated GitHub file:', response.json()['content']['html_url'])